In [1]:
import pandas as pd
import polars as pl
from pathlib import Path
import util

pd.set_option('display.float_format', '{:,.0f}'.format)

In [2]:
def mic_equity_geog(df_in, equity_group):

    df_inc = df.pivot_table(
        index='person_work_mic',
        columns=equity_group,
        values='psexpfac',
        aggfunc='sum'
    ).reset_index()

    df_inc = df_inc.rename_axis(None, axis=1)
    pd.set_option('display.float_format', '{:,.0f}'.format)
    df_inc['Percent in Equity Geog'] = df_inc[1] / (df_inc[0] + df_inc[1])
    df_inc['Percent in Equity Geog'] = df_inc['Percent in Equity Geog'].fillna(0)
    df_inc['Percent in Equity Geog'] = (df_inc['Percent in Equity Geog'] * 100).round(1).astype(str) + '%'
    df_inc.sort_values(by=0, ascending=False)

    df_inc.rename(columns={'person_work_mic': 'MIC Work Location',
                        0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population'}, inplace=True)
    df_inc.index = df_inc['MIC Work Location']
    df_inc.drop('MIC Work Location', axis=1, inplace=True)
    # Percent of people of color
    
    return df_inc

## Workers in Manufacturing-Industrial Centers (MICs)

In [3]:
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')

In [4]:
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')
df = df[df['pwtyp'].isin(['Paid Full-Time Worker', 'Paid Part-Time Worker'])]
df.loc[df['person_work_mic'].isnull(), 'person_work_mic'] = 'Outside MIC'
_df = df[['psexpfac','person_work_mic']].groupby('person_work_mic').sum()[['psexpfac']]
_df.index
_df.rename(columns={'psexpfac': 'Total Workers'}, inplace=True)
_df


,Total Workers
person_work_mic,
Ballard-Interbay,"18,805"
Cascade,"16,771"
Duwamish,"62,933"
Frederickson,"5,532"
Kent MIC,"42,004"
North Tukwila,"4,758"
Outside MIC,"2,395,562"
Paine Field / Boeing Everett,"38,596"
Port of Tacoma,"10,413"


In [5]:
mic_equity_geog(df, "hh_efa_pov200").loc[_df.index]

,Below Regional Average,Above Regional Average,Higher Share of Equity Population,Percent in Equity Geog
person_work_mic,,,,
Ballard-Interbay,"13,489","3,518","1,797",20.7%
Cascade,"9,336","5,269","2,161",36.1%
Duwamish,"36,127","16,690","10,116",31.6%
Frederickson,"2,710","1,933",889,41.6%
Kent MIC,"20,119","12,965","8,920",39.2%
North Tukwila,"2,302","1,473",983,39.0%
Outside MIC,"1,433,254","615,433","346,797",30.0%
Paine Field / Boeing Everett,"21,453","10,619","6,516",33.1%
Port of Tacoma,"4,396","3,377","2,640",43.4%


In [6]:
mic_equity_geog(df, "hh_efa_poc").loc[_df.index]

,Below Regional Average,Above Regional Average,Higher Share of Equity Population,Percent in Equity Geog
person_work_mic,,,,
Ballard-Interbay,"11,888","4,595","2,321",27.9%
Cascade,"14,725","1,742",299,10.6%
Duwamish,"27,745","18,078","17,110",39.5%
Frederickson,"2,947","1,737",848,37.1%
Kent MIC,"14,579","12,837","14,588",46.8%
North Tukwila,"1,644","1,372","1,742",45.5%
Outside MIC,"1,249,524","728,543","417,417",36.8%
Paine Field / Boeing Everett,"20,784","13,963","3,841",40.2%
Port of Tacoma,"4,896","3,525","1,992",41.9%


In [7]:
mic_equity_geog(df, "hh_efa_lep").loc[_df.index]

,Below Regional Average,Above Regional Average,Higher Share of Equity Population,Percent in Equity Geog
person_work_mic,,,,
Ballard-Interbay,"12,813","3,562","2,429",21.8%
Cascade,"14,236","1,764",766,11.0%
Duwamish,"32,722","14,414","15,797",30.6%
Frederickson,"4,281",804,447,15.8%
Kent MIC,"17,138","11,044","13,822",39.2%
North Tukwila,"1,985","1,208","1,565",37.8%
Outside MIC,"1,463,771","526,587","405,126",26.5%
Paine Field / Boeing Everett,"19,732","9,482","9,374",32.5%
Port of Tacoma,"6,907","2,129","1,377",23.6%


## Commute Characteristics

In [8]:
df = pd.read_csv(util.output_path / 'agg/dash/tour_mic_dest.csv')
# Commute tours to MICs
df = df[df['pdpurp']=="Work"]
df.loc[df['tour_d_mic'].isnull(), 'tour_d_mic'] = 'Outside MIC'

# tmodetp by tour_d_mic
df_mode = pd.pivot_table(
    df,
    index='tour_d_mic',
    columns='tmodetp',
    values='toexpfac',
    aggfunc='sum'
)

# mode share by tour_d_mic
df_mode = df_mode.div(df_mode.sum(axis=1), axis=0)
df_mode = df_mode.fillna(0)
df_mode = df_mode.reset_index()
df_mode.rename(columns={'tour_d_mic': 'MIC Work Location'}, inplace=True)
df_mode.index = df_mode['MIC Work Location']
df_mode.drop('MIC Work Location', axis=1, inplace=True)

df_mode['Transit'] = df_mode['Transit']+df_mode['Park']
df_mode['HOV'] = df_mode['HOV2'] + df_mode['HOV3+']
df_mode['Walk/Bike/Other'] =  df_mode['Walk']+df_mode['Bike']+df_mode['TNC']
# df_mode = df_mode.sort_values(by='Drive Alone', ascending=False)
# Show results as percentages
df_mode = df_mode.map(lambda x: f"{x:.1%}")
# df_mode.sort_values(by='Walk', ascending=False, inplace=True)


df_mode[['SOV', 'HOV', 'Transit', 'Walk/Bike/Other']]

tmodetp,SOV,HOV,Transit,Walk/Bike/Other
MIC Work Location,,,,
Ballard-Interbay,62.7%,25.2%,2.9%,9.1%
Cascade,66.3%,30.4%,0.2%,3.0%
Duwamish,64.7%,27.8%,4.3%,3.2%
Frederickson,67.0%,30.7%,0.4%,1.9%
Kent MIC,66.0%,30.0%,2.0%,1.9%
North Tukwila,67.3%,27.7%,2.6%,2.3%
Outside MIC,57.4%,25.7%,6.2%,10.7%
Paine Field / Boeing Everett,67.8%,29.4%,0.5%,2.2%
Port of Tacoma,66.3%,29.5%,1.6%,2.7%


In [9]:
# Commute distance
pd.set_option('display.float_format', '{:,.1f}'.format)

df = pd.read_csv(util.output_path / 'agg/dash/tour_distance_mic.csv')
df = df[df['pdpurp'] == 'Work']
df['wt_dist'] = df['tautodist_bin'] * df['toexpfac']
df = df.groupby("person_work_mic").agg(
    {'wt_dist': 'sum', 'toexpfac': 'sum'}
)
df['average_distance'] = df['wt_dist']/df['toexpfac']
df[['average_distance']]

,average_distance
person_work_mic,
Ballard-Interbay,9.9
Cascade,10.6
Duwamish,12.4
Frederickson,8.9
Kent MIC,12.5
North Tukwila,12.1
Paine Field / Boeing Everett,10.7
Port of Tacoma,11.1
Puget Sound Industrial Center- Bremerton,12.2


## Demographics

In [10]:
# Demographics 
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')
df['income_wt'] = df['hhincome_thousands'] * df['psexpfac']
df['income_wt'] = df['income_wt'].fillna(0)
df = df.groupby('person_work_mic').agg(
    {'psexpfac': 'sum', 'income_wt': 'sum'}
).reset_index()
df["avg_weighted_hh_income"] = df['income_wt']/df['psexpfac']
df['avg_weighted_hh_income'] = df['avg_weighted_hh_income'].apply(lambda x: f"${x:,.0f}")
df[['person_work_mic', 'avg_weighted_hh_income']]

,person_work_mic,avg_weighted_hh_income
0,Ballard-Interbay,"$187,221"
1,Cascade,"$153,193"
2,Duwamish,"$179,366"
3,Frederickson,"$140,539"
4,Kent MIC,"$158,447"
5,North Tukwila,"$154,922"
6,Paine Field / Boeing Everett,"$154,185"
7,Port of Tacoma,"$143,752"
8,Puget Sound Industrial Center- Bremerton,"$136,750"
9,Sumner Pacific,"$144,650"


In [11]:
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')
df['hhsize_wt'] = df['hhsize'] * df['psexpfac']
df['hhsize_wt'] = df['hhsize_wt'].fillna(0)
df = df.groupby('person_work_mic').agg(
    {'psexpfac': 'sum', 'hhsize_wt': 'sum'}
)
df['avg_wt_hhsize'] = df['hhsize_wt']/df['psexpfac']
df = df.reset_index()
df[['person_work_mic', 'avg_wt_hhsize']]

,person_work_mic,avg_wt_hhsize
0,Ballard-Interbay,2.8
1,Cascade,3.3
2,Duwamish,3.0
3,Frederickson,3.4
4,Kent MIC,3.3
5,North Tukwila,3.1
6,Paine Field / Boeing Everett,3.1
7,Port of Tacoma,3.2
8,Puget Sound Industrial Center- Bremerton,3.0
9,Sumner Pacific,3.3


In [12]:
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')
# Pivot table of worker type by person_work_mic
df_worker_type = df.pivot_table(
    index='person_work_mic',
    columns='pwtyp',
    values='psexpfac',
    aggfunc='sum'
)

# Share by worker type

df_worker_type = df_worker_type.div(df_worker_type.sum(axis=1), axis=0)
df_worker_type.rename(columns={'Paid Full-Time Worker': 'Full Time', 'Paid Part-Time Worker': 'Part Time'})
df_worker_type.map(lambda x: f"{x:.1%}")

pwtyp,Paid Full-Time Worker,Paid Part-Time Worker
person_work_mic,,
Ballard-Interbay,83.3%,16.7%
Cascade,77.3%,22.7%
Duwamish,86.0%,14.0%
Frederickson,68.0%,32.0%
Kent MIC,86.8%,13.2%
North Tukwila,79.4%,20.6%
Paine Field / Boeing Everett,82.7%,17.3%
Port of Tacoma,83.5%,16.5%
Puget Sound Industrial Center- Bremerton,66.7%,33.3%
